# GLIDE multi-site validation — all sites vs NAME & FLEXPART

This run (`outputs/icos-validation`) launched **56 validation sites** simultaneously
(`multi_point_periodic`), 48 hourly releases each (2024-01-01 00h → 2024-01-02 23h),
on the EUROPE grid. This notebook:

1. plots **all sites' footprints in one combined map** (network coverage), then
2. converts each site's GLIDE footprints to CH₄ enhancement (EDGAR 2024 flux) and
   **compares against the NAME and FLEXPART validation timeseries** — both as an
   all-sites scatter and a per-site timeseries drill-down.

It reuses the same physics bridge as `flexpart_comparison.ipynb`
(`lpdm.comparison.to_stilt_surface_footprint`: raw residence-time → STILT surface
sensitivity `(mol/mol)/(mol m⁻² s⁻¹)`), applied across the multi-site `release` axis.

In [ ]:
from __future__ import annotations

from functools import reduce
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

import holoviews as hv
import geoviews as gv
import hvplot.xarray   # noqa: F401  registers .hvplot on DataArray
import hvplot.pandas   # noqa: F401  registers .hvplot on DataFrame

from lpdm.comparison import to_stilt_surface_footprint

hv.extension("bokeh")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
GLIDE_ZARR = PROJECT_ROOT / "outputs" / "icos-validation" / "footprints.zarr"
EDGAR_NC   = PROJECT_ROOT / "data" / "EDGAR_CH4_2024_MHD_grid.nc"
VTS_DIR    = PROJECT_ROOT / "data" / "validation-timeseries"

# Must match the FLEXPART/NAME surface layer and the GLIDE bottom z-bin (both 0–40 m)
# for an exact (non-approximated) conversion.
SURFACE_LAYER_DEPTH_M = 40.0
# Representative near-surface air density for the raw→STILT conversion. The enhancement
# scales linearly with this; a single value matches flexpart_comparison.ipynb's full-
# GLIDE-series convention (per-release met p,T would make it exact).
AIR_DENSITY_KG_M3 = 1.2

print(f"GLIDE store: {GLIDE_ZARR}  (exists: {GLIDE_ZARR.exists()})")
print(f"EDGAR flux:  {EDGAR_NC}  (exists: {EDGAR_NC.exists()})")
print(f"validation:  {VTS_DIR}  (exists: {VTS_DIR.exists()})")

## Load the multi-site store

The footprint store has a flat `release` axis (one entry per site × time) carrying
`site`, `release_time`, `release_lon`, `release_lat` coords. EDGAR is snapped onto the
footprint grid (they match to ~1e-14, but xarray aligns on *exactly* equal labels, so a
bare product would silently zero all but bit-identical cells).

In [ ]:
glide_ds = xr.open_zarr(GLIDE_ZARR)
glide_fp = glide_ds["footprint"]            # (release, time_ago, z_bin, lat, lon)

sites = glide_fp["site"].values
rel_times = glide_fp["release_time"].values
uniq_sites = sorted(set(sites.tolist()))
uniq_times = np.unique(rel_times)

# One (lon, lat) per site for the map markers.
site_loc = (
    pd.DataFrame({
        "site": sites,
        "lon": glide_fp["release_lon"].values,
        "lat": glide_fp["release_lat"].values,
    })
    .drop_duplicates("site")
    .set_index("site")
    .loc[uniq_sites]
)

edgar_flux = xr.open_dataset(EDGAR_NC)["flux"]      # (lat, lon), mol m-2 s-1
edgar_flux = edgar_flux.assign_coords(
    latitude=glide_fp["latitude"].values, longitude=glide_fp["longitude"].values
)

print(f"{len(uniq_sites)} sites × {len(uniq_times)} release times = {glide_fp.sizes['release']} footprints")
print(f"window: {np.datetime_as_string(uniq_times.min(), unit='h')} .. "
      f"{np.datetime_as_string(uniq_times.max(), unit='h')}")

## All sites in one image — combined footprint at one instant

Pick a single release time and sum the raw residence-time footprints over **all sites at
that instant**, then convert to STILT surface sensitivity. This shows where the whole
network was sensitive at that hour; the cyan dots are the 56 release sites. Change
`TIME_INDEX` (0 … {n−1}) to step through the 48-hour window. (The conversion is linear,
so summing raw then converting equals converting then summing for a constant air density.)

In [ ]:
TIME_INDEX = 0                          # index into uniq_times (0 .. len-1)
snapshot_time = uniq_times[TIME_INDEX]

# All sites released at this instant -> sum over that subset of the release axis.
# (rel_times is the materialised numpy coord; the store's is dask-backed, so we
# build the integer index here rather than mask a dask array.)
at_t = glide_fp.isel(release=np.where(rel_times == snapshot_time)[0])
combined_raw = at_t.sum("release")      # (time_ago, z_bin, lat, lon)
combined_stilt = to_stilt_surface_footprint(
    combined_raw,
    surface_layer_depth_m=SURFACE_LAYER_DEPTH_M,
    air_density_kg_m3=AIR_DENSITY_KG_M3,
    integrate_time=True,
).compute()                             # (lat, lon)

site_markers = gv.Points(
    [(r.lon, r.lat) for r in site_loc.itertuples()]
).opts(color="cyan", size=7, line_color="black")

stamp = np.datetime_as_string(snapshot_time, unit="h")
combined_map = (
    combined_stilt.where(combined_stilt > 0).hvplot.image(
        x="longitude", y="latitude",
        tiles="CartoLight", cmap="magma", logz=True,
        geo=True, project=False, height=620, width=820, alpha=0.85,
        colorbar=True,
        clabel="combined surface sensitivity (mol/mol)/(mol/m²/s)",
        title=f"GLIDE combined footprint — {at_t.sizes['release']} sites @ {stamp}",
    )
    * site_markers
)
combined_map

## GLIDE CH₄ enhancement, every site

Convolve each release's surface footprint with the EDGAR flux and sum over the domain
→ ΔCH₄ (ppb) per (site, time):

$$\Delta c(t) = 10^9 \sum_{i,j} f_{ij}(t)\,q_{ij}$$

`to_stilt_surface_footprint` only accepts a single 4-D footprint, so to convert all 2688
releases at once we apply its formula **vectorised** over the `release` axis. Here the
bottom z-bin is exactly the 0–40 m surface layer, so the STILT conversion collapses to
selecting that bin × `m_air / (h ρ)` — we assert this matches the library on one release
so the vectorised path stays faithful to the reference machinery.

In [ ]:
M_AIR_KG_PER_MOL = 0.02897  # to_stilt_surface_footprint default

# Surface z-bin must coincide with the STILT surface layer for the exact (non-overlap-
# approximated) conversion; the run is configured this way (bottom bin 0–40 m).
surf_mask = (glide_fp["z_bottom_m"].values == 0.0) & (
    glide_fp["z_top_m"].values == SURFACE_LAYER_DEPTH_M
)
assert surf_mask.sum() == 1, "no single z-bin matches the 0–40 m surface layer"
z_surf = int(np.where(surf_mask)[0][0])
stilt_factor = M_AIR_KG_PER_MOL / (SURFACE_LAYER_DEPTH_M * AIR_DENSITY_KG_M3)

# (release, lat, lon) STILT surface sensitivity, vectorised over release.
glide_stilt = glide_fp.isel(z_bin=z_surf).sum("time_ago") * stilt_factor

# Faithfulness check: vectorised path == library on release 0.
_lib0 = to_stilt_surface_footprint(
    glide_fp.isel(release=0).compute(),
    surface_layer_depth_m=SURFACE_LAYER_DEPTH_M,
    air_density_kg_m3=AIR_DENSITY_KG_M3, integrate_time=True,
)
assert np.allclose(glide_stilt.isel(release=0).values, _lib0.values, atol=1e-6), \
    "vectorised STILT conversion diverges from to_stilt_surface_footprint"

glide_ppb = (glide_stilt * edgar_flux).sum(["latitude", "longitude"]).compute() * 1e9

glide_long = pd.DataFrame({
    "site": sites,
    "time": rel_times.astype("datetime64[ns]"),
    "glide_ppb": glide_ppb.values,
})
print(f"GLIDE enhancement: {len(glide_long)} (site,time) points")
print(glide_long.groupby("site")["glide_ppb"].mean().describe().round(2).to_string())

## Load NAME & FLEXPART validation timeseries

The validation CSVs are full-year hourly; restrict to this run's 48-hour window so the
comparison is like-for-like. Each site may have FLEXPART (ECMWF-HRES) and/or NAME (UMG,
UKV) runs.

In [ ]:
def load_validation(site_label: str) -> dict[str, pd.DataFrame]:
    # {model-driver: DataFrame} for one site, e.g. 'FLEXPART-ECMWFHRES', 'NAME-UMG'.
    runs = {}
    for path in sorted(VTS_DIR.glob(f"{site_label}_*.csv")):
        name = path.stem[len(site_label) + 1:].replace("_", "-")
        df = pd.read_csv(path, comment="#")
        df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_localize(None)
        runs[name] = df
    return runs

t0, t1 = glide_long["time"].min(), glide_long["time"].max()

# Long table of every model's enhancement within the GLIDE window, all sites.
val_rows = []
for site in uniq_sites:
    for model, df in load_validation(site).items():
        win = df[(df["time"] >= t0) & (df["time"] <= t1)]
        if len(win):
            val_rows.append(win.assign(site=site, model=model))
val_long = (
    pd.concat(val_rows, ignore_index=True)
    .rename(columns={"ch4_enhancement_ppb": "val_ppb"})[["site", "model", "time", "val_ppb"]]
)
print("validation points in window by model:")
print(val_long.groupby("model")["site"].nunique().to_string())

## All-sites comparison — scatter vs NAME & FLEXPART

One point per site: mean ΔCH₄ over the 48-hour window, GLIDE (x) vs each reference model
(y). The dashed line is 1:1. Points on the line mean GLIDE and that model agree on the
site's mean sensitivity to EDGAR emissions.

In [ ]:
glide_site_mean = glide_long.groupby("site")["glide_ppb"].mean()
val_site_mean = val_long.groupby(["model", "site"])["val_ppb"].mean().reset_index()
val_site_mean["glide_ppb"] = val_site_mean["site"].map(glide_site_mean)

MODEL_COLORS = {
    "FLEXPART-ECMWFHRES": "#3182bd",
    "NAME-UMG":           "#31a354",
    "NAME-UKV":           "#756bb1",
}

hi = float(np.nanmax([glide_site_mean.max(), val_site_mean["val_ppb"].max()])) * 1.05
one_to_one = hv.Curve([(0, 0), (hi, hi)]).opts(color="black", line_dash="dashed", line_width=1)

scatter = reduce(lambda a, b: a * b, [
    sub.hvplot.scatter(
        x="glide_ppb", y="val_ppb", label=model, color=MODEL_COLORS.get(model),
        hover_cols=["site"], size=45,
    )
    for model, sub in val_site_mean.groupby("model")
])

(one_to_one * scatter).opts(
    xlabel="GLIDE mean ΔCH₄ (ppb)", ylabel="reference mean ΔCH₄ (ppb)",
    title="48-h mean CH₄ enhancement: GLIDE vs NAME / FLEXPART (one point per site)",
    width=640, height=620, legend_position="top_left", xlim=(0, hi), ylim=(0, hi),
)

## Per-site drill-down timeseries

Set `SITE_LABEL` to any of the 56 inlets to overlay GLIDE against every reference run for
that site over the 48-hour window.

In [ ]:
SITE_LABEL = "MHD-10magl"   # any site in uniq_sites

g = glide_long[glide_long["site"] == SITE_LABEL].sort_values("time")
lines = [g.hvplot.line(x="time", y="glide_ppb", label="GLIDE",
                       color="#e6550d", line_width=2.5)]
for model, df in val_long[val_long["site"] == SITE_LABEL].groupby("model"):
    lines.append(df.sort_values("time").hvplot.line(
        x="time", y="val_ppb", label=model, color=MODEL_COLORS.get(model), line_width=1.6))

reduce(lambda a, b: a * b, lines).opts(
    xlabel="Time (UTC)", ylabel="ΔCH₄ (ppb)",
    title=f"CH₄ enhancement at {SITE_LABEL} — GLIDE vs reference LPDMs",
    legend_position="top_right", height=430, width=1000,
)